# Voice fine-tune (Piper) — brief §8.5

**EXPERIMENTAL, untested, fragile.** First fine-tune attempt on the founder's
voice, using the dataset from `ingest.ipynb`. Piper chosen for: MIT licence,
offline, low VRAM, the most documented fine-tune flow (brief §2.5 "simple over
impressive"). If prosody comes out too flat we escalate to an XTTS/F5 fine-tune
on a local GPU box.

Prereqs: you ran `ingest.ipynb` and saved `dataset_v1.zip` to
`MyDrive/voice_training/`, and its transcripts look right.

`Runtime → T4 GPU`, run top to bottom. Training cell 7 is the long one — set
`MAX_EPOCHS` for how long you can leave Colab running.


In [ ]:
#@title 1. GPU check
!nvidia-smi -L || print('NO GPU — Runtime > Change runtime type > T4 GPU')


In [ ]:
#@title 2. Install Piper training stack (~4-6 min, may print pip warnings)
!apt-get -qq install espeak-ng > /dev/null
!git clone -q https://github.com/rhasspy/piper /content/piper
%cd /content/piper/src/python
!pip -q install -e .
!pip -q install "pytorch-lightning~=1.9" "torchmetrics==0.11.4" onnx onnxruntime piper-tts
!bash build_monotonic_align.sh
print('done')


In [ ]:
%%writefile /content/benchmark_sentences.json
{
  "_comment": "Machine-readable copy of TANZANIA_VOICE_BENCHMARK.md test sentences. Keep the two in sync. Used by worker/colab/voice_worker.py --mode benchmark.",
  "language": "sw-TZ",
  "sentences": [
    { "id": "A1", "category": "normal", "text": "Habari za leo? Mimi niko poa tu, nashukuru Mungu." },
    { "id": "A2", "category": "normal", "text": "Leo tutaangalia jinsi ya kutumia hii app kwa haraka na bila usumbufu." },
    { "id": "A3", "category": "normal", "text": "Karibu tena kwenye channel yangu, nafurahi umerudi." },
    { "id": "B4", "category": "excited", "text": "Jamani, hii habari ni kubwa sana — lazima muione video hii mpaka mwisho!" },
    { "id": "B5", "category": "excited", "text": "Nimefurahi kweli kuwaonyesha kitu kilichobadilisha biashara yangu kabisa!" },
    { "id": "B6", "category": "excited", "text": "Hii ndio siku tuliyokuwa tunaisubiri, na sasa imefika!" },
    { "id": "C7", "category": "professional", "text": "Katika taarifa ya leo, tutajadili matokeo ya robo mwaka na mpango wa mbele." },
    { "id": "C8", "category": "professional", "text": "Tunawashukuru wateja wetu wote kwa kutuamini kwa mwaka mzima huu." },
    { "id": "C9", "category": "professional", "text": "Timu yetu imefanya kazi kwa bidii kuhakikisha huduma inaboreka." },
    { "id": "D10", "category": "code-switch", "text": "Ukishafungua dashboard, bonyeza kitufe cha login halafu weka password yako." },
    { "id": "D11", "category": "code-switch", "text": "Content yako lazima iwe na thumbnail nzuri ili watu waweze ku-click." },
    { "id": "D12", "category": "code-switch", "text": "Nili-post reel moja TikTok jana, ika-hit views elfu hamsini ndani ya masaa manne." },
    { "id": "D13", "category": "code-switch", "text": "Tumia hashtag sahihi, halafu uangalie analytics baada ya siku mbili." },
    { "id": "E14", "category": "storytelling", "text": "Siku moja, nikiwa bado nauza nyanya sokoni, sikujua kama maisha yangu yangebadilika." },
    { "id": "E15", "category": "storytelling", "text": "Nakumbuka usiku ule, mvua ikinyesha, nikiandika mpango wangu wa kwanza wa biashara." },
    { "id": "E16", "category": "storytelling", "text": "Nilianza na mtaji wa shilingi elfu kumi tu, na watu wengi walinicheka." },
    { "id": "F17", "category": "sales-cta", "text": "Usikose nafasi hii — bonyeza link iliyoko chini na ujiunge leo kabla haijaisha." },
    { "id": "F18", "category": "sales-cta", "text": "Bei ya kawaida ni laki mbili, lakini leo tu utapata kwa laki moja na nusu." },
    { "id": "F19", "category": "sales-cta", "text": "Wateja wa kwanza hamsini watapata bonus ya ziada, kwa hiyo usichelewe." },
    { "id": "G20", "category": "educational", "text": "Riba rahisi hukokotolewa kwa kuzidisha kiasi cha msingi, kiwango cha riba, na muda." },
    { "id": "G21", "category": "educational", "text": "Kuna aina tatu za bajeti: ya kila siku, ya kila mwezi, na ya mwaka mzima." },
    { "id": "G22", "category": "educational", "text": "Akiba ni pesa unayoweka pembeni kabla hujaanza kutumia mapato yako." },
    { "id": "H23", "category": "pronunciation", "text": "Ng'ombe wangu wamekwenda kunywa maji mtoni asubuhi na mapema." },
    { "id": "H24", "category": "pronunciation", "text": "Mzee Mng'ong'o alinunua ng'ombe wawili na mbuzi wenye nguvu." },
    { "id": "H25", "category": "pronunciation", "text": "Nyanya, mchicha, na mboga za majani ni muhimu kwa afya ya mwili." },
    { "id": "H26", "category": "pronunciation", "text": "Ndugu zangu, mnaokaa mbali, karibuni nyumbani kwa sherehe." },
    { "id": "H27", "category": "pronunciation", "text": "Mwalimu alimwambia mwanafunzi aandike insha kuhusu mvua na ukame." }
  ]
}


In [ ]:
#@title 3. Get the dataset from Drive
from google.colab import drive
drive.mount('/content/drive')
ZIP = '/content/drive/MyDrive/voice_training/dataset_v1.zip'  #@param {type:"string"}
import shutil, os, pathlib
os.makedirs('/content/ds', exist_ok=True)
shutil.unpack_archive(ZIP, '/content/ds')
# ingest saved metadata as 'wavs/<id>.wav|text'; piper wants '<id>|text'
src = pathlib.Path('/content/ds/dataset/metadata.csv')
rows = [l.rstrip('\n') for l in src.open(encoding='utf-8') if '|' in l]
fixed = []
for l in rows:
    w, t = l.split('|', 1)
    fixed.append(f"{pathlib.Path(w).stem}|{t}")
src.write_text('\n'.join(fixed) + '\n', encoding='utf-8')
print(len(fixed), 'clips; sample:', fixed[0][:80])


In [ ]:
#@title 4. Preprocess
!python -m piper_train.preprocess \
  --language sw --input-dir /content/ds/dataset --output-dir /content/train \
  --dataset-format ljspeech --single-speaker --sample-rate 22050


In [ ]:
#@title 5. Base checkpoint to fine-tune from
#@markdown Cross-language fine-tune works, it just needs more epochs. If a
#@markdown Swahili checkpoint exists in the repo below, prefer it.
BASE_URL = "https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/en/en_US/lessac/medium/epoch%3D2164-step%3D1355540.ckpt"  #@param {type:"string"}
!wget -q -O /content/base.ckpt "$BASE_URL" && ls -lh /content/base.ckpt


In [ ]:
#@title 6. Fine-tune  (the long cell)
MAX_EPOCHS = 2000  #@param {type:"integer"}
BATCH_SIZE = 16     #@param {type:"integer"}
!python -m piper_train \
  --dataset-dir /content/train \
  --accelerator gpu --devices 1 \
  --batch-size $BATCH_SIZE \
  --validation-split 0.0 --num-test-examples 0 \
  --max_epochs $MAX_EPOCHS \
  --resume_from_checkpoint /content/base.ckpt \
  --checkpoint-epochs 250 \
  --precision 32


In [ ]:
#@title 7. Export to ONNX
import glob
ck = sorted(glob.glob('/content/train/lightning_logs/version_*/checkpoints/*.ckpt'))[-1]
print('exporting', ck)
!python -m piper_train.export_onnx "$ck" /content/founder.onnx
!cp /content/train/config.json /content/founder.onnx.json
!ls -lh /content/founder.onnx*


In [ ]:
#@title 8. Synthesize the 27 benchmark sentences + listen
import json, subprocess, pathlib, IPython.display as ipd
S = json.load(open('/content/benchmark_sentences.json'))['sentences']
out = pathlib.Path('/content/out/piper_ft'); out.mkdir(parents=True, exist_ok=True)
for s in S:
    f = out / f"{s['id']}_{s['category']}.wav"
    subprocess.run(['piper','-m','/content/founder.onnx','-f',str(f)],
                   input=s['text'].encode(), check=True)
for f in sorted(out.glob('*.wav')):
    print(f.name); ipd.display(ipd.Audio(str(f)))


## Score it — `TANZANIA_VOICE_BENCHMARK.md`

Compare against the MMS baseline: is it more **you**? Tanzanian accent better,
same, or worse? Still robotic? `ng'ombe` / `mchicha` right?

- If clearly better and near §4 → we integrate it behind `VoiceProvider`.
- If your voice is there but Swahili is off → more/cleaner data, or escalate to
  an XTTS/F5 fine-tune (local GPU).
- If flat/robotic → Piper's ceiling; escalate.

Keep this as **Founder Voice v1** (brief §8.5 versioning) — never overwrite it.


In [ ]:
#@title 9. Save the model + samples to Drive
import shutil
shutil.copy('/content/founder.onnx', '/content/drive/MyDrive/voice_training/founder_voice_v1.onnx')
shutil.copy('/content/founder.onnx.json', '/content/drive/MyDrive/voice_training/founder_voice_v1.onnx.json')
shutil.make_archive('/content/drive/MyDrive/voice_training/piper_ft_v1_samples', 'zip', '/content/out/piper_ft')
print('saved to MyDrive/voice_training/')
